In [0]:
CREATE OR REPLACE VIEW workspace.dimension.dim_date AS
-- ===================================================================
-- Create logic for date dimension
-- ===================================================================
SELECT
    CAST(DATE_FORMAT(DATE_ID, 'yyyyMMdd') AS INT) AS DATE_ID
  , DATE_ID AS DATE
  -- Calendar parts
  , DAY(DATE_ID) AS DAY_OF_MONTH
  , MONTH(DATE_ID) AS MONTH
  , YEAR(DATE_ID) AS YEAR
  -- Month names
  , DATE_FORMAT(DATE_ID, 'MMM') AS MONTH_NAME_SHORT
  , DATE_FORMAT(DATE_ID, 'MMMM') AS MONTH_NAME_FULL
  -- ISO weekday / week / week-year
  , EXTRACT(DAYOFWEEK_ISO FROM DATE_ID) AS DAY_OF_WEEK_NUM
  , UPPER(DATE_FORMAT(DATE_ID, 'E')) AS DAY_OF_WEEK_DESC
  , WEEKOFYEAR(DATE_ID) AS ISO_WEEK
  , CONCAT(WEEKOFYEAR(DATE_ID), '-', EXTRACT(DAYOFWEEK_ISO FROM DATE_ID)) AS SAME_DAY_KEY
  -- Week boundaries
  , CAST(DATE_TRUNC('WEEK', DATE_ID) AS DATE) AS START_OF_WEEK
  , DATE_ADD(CAST(DATE_TRUNC('WEEK', DATE_ID) AS DATE), 6) AS END_OF_WEEK
  -- Quarter
  , QUARTER(DATE_ID) AS QUARTER
  -- Financial year (FY rolls in July)
  , YEAR(DATE_ID) + CASE WHEN MONTH(DATE_ID) >= 7 THEN 1 ELSE 0 END AS FIN_YEAR
  , CONCAT('FY', YEAR(DATE_ID) + CASE WHEN MONTH(DATE_ID) >= 7 THEN 1 ELSE 0 END) AS FIN_YEAR_DESC
  -- ISO year-of-week
  , EXTRACT(YEAROFWEEK FROM DATE_ID) AS ISO_YEAR
  -- Display fields
  , DATE_FORMAT(DATE_ID, 'MMM-yyyy') AS MONTH_YEAR
  -- Offsets
  , DATE_DIFF(DAY, DATE_ID, CURRENT_DATE()) * -1 AS DAY_OFFSET
  , FLOOR(DATEDIFF(DAY, DATE_TRUNC('WEEK', DATE_ID), DATE_TRUNC('WEEK', CURRENT_DATE())) / 7) * -1 AS WEEK_OFFSET
  , (year(DATE_ID) - year(current_date())) * 12 
    + (month(DATE_ID) - month(current_date())) AS MONTH_OFFSET
  , YEAR(DATE_ID) - YEAR(CURRENT_DATE()) AS YEAR_OFFSET
	, CASE WHEN (CASE WHEN MONTH(DATE_ID) >= 7 THEN YEAR(DATE_ID) + 1 ELSE YEAR(DATE_ID) END) 
      	=
      	(CASE WHEN MONTH(CURRENT_DATE) >= 7 THEN YEAR(CURRENT_DATE) + 1 ELSE YEAR(CURRENT_DATE) END)
      	THEN 0
      	WHEN (CASE WHEN MONTH(DATE_ID) >= 7 THEN YEAR(DATE_ID) + 1 ELSE YEAR(DATE_ID) END) 
      	=
      	(CASE WHEN MONTH(CURRENT_DATE) >= 7 THEN YEAR(CURRENT_DATE) + 1 ELSE YEAR(CURRENT_DATE) END) - 1
      	THEN - 1
      ELSE 99
      END AS FIN_YEAR_OFFSET
FROM workspace.reference.dates
WHERE 1=1
--  AND DATE_ID <= DATE_ADD(CURRENT_DATE(), -1) -- historical only (<= yesterday)
;



select *
from workspace.reference.dates
